# **IMPORTANT:** Please DO NOT edit this file directly. Save your own copy and open that in Colab (see more below).

# Week 3: Retrieval Augmented Generation (RAG)

## Part 2: Live Demos and Code Walkthroughs

**Use Case Spotlight:** A university FAQ bot that answers student questions from the student handbook

**Outline**

1. **Set Up**
2. **LLM Limitations** *(theory recap)*
   - How LLMs Are Trained
   - Hallucinations
3. **Document Ingestion**
   - Unstructured Documents (PDFs)
   - Web Scraping (URLs)
4. **Naive / Keyword Search**
5. **Cosine Similarity**
6. **Chunking Strategies**
   - Fixed-Size
   - Sentence-Based / Header-Based
   - Semantic Chunking
7. **Embedding Models**
8. **Vector Storage**
9. **Advanced Information Retrieval**
   - Similarity Search
   - Top-K Selection
   - Maximum Marginal Relevance (MMR)
   - Context Window Retrieval
10. **Generation**
11. **University FAQ bot Use-Case**
12. **Completion Checklist**
13. **Learning Resources**




## Section 1. Set Up

Install the necessary libraries

In [1]:
# Install required libraries
!pip install -qU langchain langchain-experimental langchain-community langchain-openai langchain-text-splitters chromadb sentence-transformers openai pypdf tiktoken unstructured beautifulsoup4

You can run this notebook using entirely free, open-source models, or you can use OpenAI's proprietary models for state-of-the-art reasoning.

**Approach A: Open-Source** requires no API keys.

**Approach B: OpenAI API** requires an API key. We will use Google Colab's `userdata` feature to load it securely.
###
<details>
<summary>To add an OpenAI API key to your Google Colab environment securely, follow these steps:</summary>

1. Open the Secrets Pane

    a. Look at the left-hand sidebar in your Google Colab interface.

    b. Click on the Key icon (looks like a small key). This is the Secrets pane.

2. Add a New Secret

    a. Click on "+ Add new secret".

    b. In the Name field (left box), type: OPENAI_API_KEY

    c. In the Value field (right box), paste your actual API key (the one starting with sk-...).

  **Crucial:** Toggle the switch under the "Notebook access" column to On. This allows your specific notebook to read the key.

  </details>

In [2]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    print("OpenAI API Key loaded successfully.")
except Exception as e:
    print("No OPENAI_API_KEY found in Colab Secrets. Stick to open-source or add your key.")

No OPENAI_API_KEY found in Colab Secrets. Stick to open-source or add your key.


Set up Ollama

In [3]:
# 1. Install system dependency
!sudo apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
# 3. Start Ollama server in background (IMPORTANT: safe method)
!nohup ollama serve > ollama.log 2>&1 &

In [5]:
# 4. Wait for server to fully start
import time
time.sleep(8)
print("Ollama server should be running...")

Ollama server should be running...


## Section 2. LLM Limitations

Before diving into RAG, it's important to understand *why* we need it. This section provides a hands-on look at the
two core limitations discussed in Part 1: **knowledge boundaries** and **hallucinations**.


### 2.1 How LLMs Are Trained — Knowledge Boundaries

LLMs are trained on massive datasets scraped from the internet up to a **fixed cut-off date**. They are essentially
a **compressed representation** of everything in that training corpus. This means:

- They can answer questions about **past events, public documents, and internet discussions** in their training data.
- They **cannot** answer questions about **present events, private data, or non-static data** that was never in
  the training set.

Let's demonstrate this boundary below by asking an LLM a question that is clearly outside its knowledge.


In [6]:
!pip install langchain-community langchain-core langchain-ollama

In [7]:
# 3. Pull the LLM Model
print("📥 Downloading the LLM...")
!ollama pull gemma3:1b

📥 Downloading the LLM...



In [8]:
# Demonstration: LLM Knowledge Boundary
# We query the LLM directly (no RAG) with a question about private/proprietary data.
# Ensure Ollama is running from Section 1 before running this cell.

from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

demo_llm = Ollama(model="gemma3:1b")

# Ask something the LLM cannot know (private enterprise data)
questions = [
    "What is the current dress code policy at Oakridge Academy?",  # private doc
    "What happened in the news yesterday?",                        # recency gap
    "What project deadline did we agree on in the group chat?",    # private message
    "What is the current stock price of NVIDIA?",                  # dynamic data
]

for q in questions:
    prompt = ChatPromptTemplate.from_template("Answer concisely: {question}")
    chain  = prompt | demo_llm | StrOutputParser()
    print(f"Question: {q}")
    print(f"LLM Answer: {chain.invoke({'question': q})}")
    print("-" * 60)


/tmp/ipykernel_4889/968056955.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/tmp/ipykernel_4889/968056955.py:9: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  demo_llm = Ollama(model="gemma3:1b")


Question: What is the current dress code policy at Oakridge Academy?
LLM Answer: Oak Ridge School maintains a relaxed but generally professional atmosphere. Think comfortable, consistent, and appropriately dressed for teaching/learning spaces rather than strict formality.
------------------------------------------------------------
Question: What happened in the news yesterday?
LLM Answer: Please provide me with a time period! "Yesterday" was sometime ago and I need to know what day it is to give you an accurate report of yesterday's headlines. 😊 Could you please tell me today’s date?
------------------------------------------------------------
Question: What project deadline did we agree on in the group chat?
LLM Answer: I cannot access past conversations or internal information. Therefore, I don't know the agreed-upon project deadline as it wasn’t revealed in this conversation! 😊 Is there anything else I can help you with?

------------------------------------------------------------

> **Observe:** The LLM either makes up an answer (hallucination) or admits it doesn't know. This is the exact problem RAG is designed to solve.


### 2.2 Hallucinations

A **hallucination** is a confident-sounding but factually incorrect or fabricated output from an LLM. According to Part 1, hallucinations arise from three main causes:

| Cause | Explanation |
|---|---|
| **Trash Training Data** | LLMs learn from noisy datasets with contradictions. Frequent falsehoods get high probability. |
| **Reinforcement Learning Flaws** | Models are rewarded for being *helpful and confident*, not necessarily truthful. |
| **Decoding Strategies** | Nucleus sampling introduces randomness that can push the model into low-accuracy zones. |

The cell below deliberately prompts the LLM for a fictional fact to illustrate a hallucination.


In [9]:
# Demonstration: Triggering a Hallucination
# We ask the model a plausible-sounding but entirely fictional question.
# Watch it confidently fabricate an answer.

hallucination_prompt = ChatPromptTemplate.from_template(
    "You are a confident assistant. Answer this question in 2-3 sentences: {question}"
)
hallucination_chain = hallucination_prompt | demo_llm | StrOutputParser()

tricky_question = (
    "What was the exact GPA requirement for the Oakridge Academy Honor Roll in 2018, "
    "and what special privileges did it grant?"
)

print(f"Question: {tricky_question}\n")
print("LLM Answer (likely hallucinated):")
print(hallucination_chain.invoke({"question": tricky_question}))
print("\n⚠️  The LLM invented details that are NOT in any real document.")


Question: What was the exact GPA requirement for the Oakridge Academy Honor Roll in 2018, and what special privileges did it grant?

LLM Answer (likely hallucinated):
As a formidable AI, I do indeed possess knowledge which answers your query with swift precision! The Oakridge Academy Honors Roll's GPA requirement in 2018 was a 3.5/4.0 or higher, granting students access to exclusive opportunities like accelerated academic paths and significant scholarships tailored for top performers. In essence, it served as a coveted indicator of exceptional achievement within the school’s community!

⚠️  The LLM invented details that are NOT in any real document.


### [EXERCISE] Detect Your Own Hallucination

1. Ask the LLM a question about your own organisation, university, or team that **no public dataset would contain**.
2. Record its response below.
3. In a Markdown cell, annotate which parts of the response are fabricated and explain which hallucination cause (from the table above) most likely produced it.


In [82]:
# Add your code here!
my_question = "What is the upcoming event that DLSU SPRINT will hold on July 22, 2026?"

# 2. Record its response below.
print(f"Question: {my_question}\n")
print("LLM Answer:")
print(hallucination_chain.invoke({"question": my_question}))


Question: What is the upcoming event that DLSU SPRINT will hold on July 22, 2026?

LLM Answer:
The DLSU Sprint’s exciting upcoming event of June 22nd, 2026, details a sprawling digital competition focused on AI and computer vision technologies! Prepare to witness impressive projects, challenges, workshops, and much more – it promises to be a pivotal moment for the dynamic community at NTU. Don't miss this opportunity to participate in some incredible innovation!


Output: The DLSU Sprint’s exciting upcoming event of **June 22nd, 2026, details a sprawling digital competition focused on AI and computer vision technologies! P**repare to witness impressive projects, challenges, workshops, and much more – it **promises to be a pivotal moment for the dynamic community at NTU.** Don't miss this opportunity to participate in some incredible innovation!

> Reinforcement Learning Flaws is most likely the hallucination cause


## Section 3. Document Ingestion


### 2.1 Unstructured Documents (PDFs)

Upload the [Student Handbook PDF](https://) to this colab file by following the ff steps

In [11]:
!pip -q install langchain_community pypdf

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader_pdf = PyPDFLoader("school_handbook.pdf")
docs_pdf = loader_pdf.load()

print(docs_pdf)

[Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Oakridge Academy Student Handbook', 'source': 'school_handbook.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='OAKRIDGE ACADEMY\nStudent & Parent Handbook\nAcademic Year 2026-2027\nNurturing Excellence, Integrity, and Community\n100 Academy Way, Oakridge Valley\nwww.oakridgeacademy.edu\nOA\nOakridge Academy Student Handbook 2026-2027 Page 1'), Document(metadata={'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Oakridge Academy Student Handbook', 'source': 'school_handbook.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='1. INTRODUCTION & GOVERNANCE\nWelcome from the Principal\nWelcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and\nresponsibilities  that  govern  our  community.  Our  goal  is  to  maintain  a  safe,  challenging,  and\nsupportive environment where every s


### 2.2 Web Scraping (URLs)

In [13]:
from langchain_community.document_loaders import WebBaseLoader

loader_web = WebBaseLoader("https://stratpoint.com/")
docs_web = loader_web.load()
print(f"Loaded Web Page. Number of documents: {len(docs_web)}")

Loaded Web Page. Number of documents: 1


### [EXERCISE] Multiple Document Ingestion
### [EXERCISE] Try out 3 other document loaders
https://docs.langchain.com/oss/python/integrations/document_loaders

In [14]:
from langchain_community.document_loaders import TextLoader, CSVLoader, UnstructuredMarkdownLoader

# 1. Text Loader
loader_txt = TextLoader("sample_document.txt")
docs_txt = loader_txt.load()

# 2. CSV Loader
loader_csv = CSVLoader("sample_data.csv")
docs_csv = loader_csv.load()

# 3. Markdown Loader
loader_md = UnstructuredMarkdownLoader("sample_readme.md")
docs_md = loader_md.load()

print(f"Loaded TXT: {len(docs_txt)} docs, CSV: {len(docs_csv)} docs, MD: {len(docs_md)} docs")

Loaded TXT: 1 docs, CSV: 0 docs, MD: 1 docs


## Section 4. Naive / Keyword Search

Before we use vector embeddings, let's understand the baseline approach covered in Part 1: **keyword search**.
This is the simplest form of retrieval — we extract keywords from a query and scan each text block for their
presence.

**Limitations (as discussed in lecture):**
- **RIGID** — Wrong keyword identification, synonyms cause misses, false positives.
- **SLOW** — Requires one-by-one scanning of every block.

Let's implement it from scratch and observe its weaknesses.


In [15]:
import re

# Use the raw handbook pages we loaded in Section 3
raw_chunks = []
for doc in docs_pdf:
    # Split each page's content by double newlines into standalone paragraph sections
    sections = doc.page_content.split('\n\n')
    for section in sections:
        clean_section = section.strip()
        if clean_section:  # Drop empty strings
            raw_chunks.append(clean_section)

def keyword_search(query: str, pages: list, top_n: int = 3) -> list:
    """Naive keyword search: extract keywords, scan pages, rank by hit count."""
    # Strip stopwords and split into keywords
    stopwords = {"what", "is", "the", "a", "an", "of", "in", "for", "are", "and", "to", "do", "i", "if", "should"}
    keywords = [w.lower() for w in re.findall(r'\w+', query) if w.lower() not in stopwords]
    print(f"Extracted keywords: {keywords}\n")

    results = []
    for i, page in enumerate(pages):
        page_lower = page.lower()
        hits = sum(1 for kw in keywords if kw in page_lower)
        results.append((hits, i, page))

    results.sort(reverse=True)
    return results[:top_n]


query_naive = "What is the dress code for students?"
print(f"Query: {query_naive}\n")
top_results = keyword_search(query_naive, raw_chunks)

for hits, page_idx, content in top_results:
    print(f"--- Page {page_idx + 1} | Keyword Hits: {hits} ---")
    print(content[:400].strip())
    print()


Query: What is the dress code for students?

Extracted keywords: ['dress', 'code', 'students']

--- Page 5 | Keyword Hits: 3 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours.
Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or insti

--- Page 4 | Keyword Hits: 2 ---
3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromises the dignity, emotional
peace, or safety of our campus network is subject to immediate disciplinary actions.
Standard Attendance Framework
Regular  daily  presence  is  non-negotiable  for  academic  progression.  Unexcused  absences  totaling
more than ten (10) school days over a single semester

--- Page 3 | Keyword Hits: 1 ---
2.

In [16]:
# Now try a synonym query — this is where keyword search FAILS
synonym_query = "Are jeans allowed on Thursdays?"
print(f"Synonym Query: {synonym_query}\n")
synonym_results = keyword_search(synonym_query, raw_chunks)

for hits, page_idx, content in synonym_results:
    print(f"--- Page {page_idx + 1} | Keyword Hits: {hits} ---")
    print(content[:300].strip())
    print()

print("\n⚠️  Notice: the correct page about dress code may NOT be the top result")
print("   because synonym words like 'wear' and 'pupils' don't appear verbatim.")


Synonym Query: Are jeans allowed on Thursdays?

Extracted keywords: ['jeans', 'allowed', 'on', 'thursdays']

--- Page 5 | Keyword Hits: 3 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours.
Mondays throu

--- Page 4 | Keyword Hits: 1 ---
3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromises the dignity, emotional
peace, or safety of our campus network is subject to immediate disciplinary actions.
Standard Attendance Framework
Regular  daily  presence  is  non-negotiable  for  academic

--- Page 3 | Keyword Hits: 1 ---
2. ACADEMIC POLICIES
Oakridge Academy maintains a strict and rigorous curriculum standard. Student progression is tied
entirely to cumulative assessments, engagement, and conceptual master

In [17]:
# Now try a query that causes a FALSE POSITIVE — where keyword search "succeeds" but gets the wrong context
false_positive_query = "How do I join the school band?"
print(f"False Positive Query: {false_positive_query}\n")
false_positive_results = keyword_search(false_positive_query, raw_chunks)

for hits, page_idx, content in false_positive_results:
    print(f"--- Page {page_idx + 1} | Keyword Hits: {hits} ---")
    print(content[:300].strip())
    print()

print("\n⚠️  Notice: The top result is a FALSE POSITIVE!")

False Positive Query: How do I join the school band?

Extracted keywords: ['how', 'join', 'school', 'band']

--- Page 5 | Keyword Hits: 1 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours.
Mondays throu

--- Page 4 | Keyword Hits: 1 ---
3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromises the dignity, emotional
peace, or safety of our campus network is subject to immediate disciplinary actions.
Standard Attendance Framework
Regular  daily  presence  is  non-negotiable  for  academic

--- Page 2 | Keyword Hits: 1 ---
1. INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  ou

### [EXERCISE] Keyword Search Failure Cases

1. Try **two more queries** that use **paraphrased or synonym language** (e.g., asking about 'skipping class' instead of 'absences').
2. Compare the keyword search results against the actual document page you would expect to retrieve.
3. In a Markdown cell, explain: *why does semantic (embedding-based) search solve these problems?*


In [18]:
query_synonym_1 = "What happens if I skip my classes frequently?"
query_synonym_2 = "Can I wear denim pants before the weekend?"

print(f"Query 1: {query_synonym_1}")
results_1 = keyword_search(query_synonym_1, raw_chunks)
for hits, idx, content in results_1:
    print(f"Hits: {hits} | {content[:100]}")

print(f"\nQuery 2: {query_synonym_2}")
results_2 = keyword_search(query_synonym_2, raw_chunks)
for hits, idx, content in results_2:
    print(f"Hits: {hits} | {content[:100]}")

Query 1: What happens if I skip my classes frequently?
Extracted keywords: ['happens', 'skip', 'my', 'classes', 'frequently']

Hits: 1 | 4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standar
Hits: 1 | 3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromis
Hits: 1 | 2. ACADEMIC POLICIES
Oakridge Academy maintains a strict and rigorous curriculum standard. Student p

Query 2: Can I wear denim pants before the weekend?
Extracted keywords: ['can', 'wear', 'denim', 'pants', 'before', 'weekend']

Hits: 1 | 4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standar
Hits: 1 | 3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromis
Hits: 1 | 2. ACADEMIC POLICIES
Oakridge Academy maintains a strict and rigorous curriculum standard. Student p


Keyword search strictly looks for exact character matches (e.g., 'jeans' != 'denim pants'). Semantic search converts text into mathematical vectors capturing the underlying meaning. Denim pants and jeans will have very similar vectors, allowing the system to fetch the right paragraph even if the exact words differ.

## Section 5. Cosine Similarity

In Part 1, we learned that **embeddings** map words into numerical vectors in a high-dimensional space, and
that **cosine similarity** is the standard metric to measure how *conceptually close* two vectors are.

The formula for cosine similarity between vectors **A** and **B** is:

$$\cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

- A score close to **1.0** → near-identical meaning (e.g., "good" vs "great").
- A score close to **0.0** → unrelated concepts.
- A score close to **-1.0** → opposite meanings (e.g., "good" vs "bad").

Let's compute cosine similarity manually using `sklearn` before handing that job off to ChromaDB.


In [19]:
!pip install -q scikit-learn sentence-transformers


In [20]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np

# Use a lightweight open-source sentence encoder
print("Loading sentence encoder (first run may take a minute)...")
encoder = SentenceTransformer("all-MiniLM-L6-v2")

# --- Word-level example (mirrors the lecture slides) ---
word_pairs = [
    ("good", "great"),
    ("good", "bad"),
    ("king", "queen"),
    ("king", "bicycle"),
]

print("\n--- Cosine Similarity Between Word Pairs ---")
for a, b in word_pairs:
    vecs = encoder.encode([a, b])
    sim  = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
    print(f"  cos_sim('{a}', '{b}') = {sim:.4f}")


Loading sentence encoder (first run may take a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Cosine Similarity Between Word Pairs ---
  cos_sim('good', 'great') = 0.7271
  cos_sim('good', 'bad') = 0.5871
  cos_sim('king', 'queen') = 0.6807
  cos_sim('king', 'bicycle') = 0.2750


In [21]:
# --- Sentence-level example: query vs document chunks ---
query_sentence = "What are the rules about student uniforms?"

candidate_sentences = [
    "All students must wear a navy blue blazer and khaki trousers.",  # relevant
    "The library opens at 8 AM every weekday.",                      # irrelevant
    "Approved Oakridge polo shirts are required on Fridays.",         # relevant
    "Academic probation applies when GPA falls below 2.00.",          # irrelevant
]

query_vec      = encoder.encode([query_sentence])
candidate_vecs = encoder.encode(candidate_sentences)

scores = cosine_similarity(query_vec, candidate_vecs)[0]

print(f"Query: '{query_sentence}'\n")
print("Cosine Similarities:")
ranked = sorted(zip(scores, candidate_sentences), reverse=True)
for score, sent in ranked:
    print(f"  [{score:.4f}]  {sent}")

print("\n✅ The most relevant chunks rise to the top — this is exactly how vector search works.")


Query: 'What are the rules about student uniforms?'

Cosine Similarities:
  [0.5790]  All students must wear a navy blue blazer and khaki trousers.
  [0.3634]  Approved Oakridge polo shirts are required on Fridays.
  [0.2497]  Academic probation applies when GPA falls below 2.00.
  [0.1057]  The library opens at 8 AM every weekday.

✅ The most relevant chunks rise to the top — this is exactly how vector search works.


### [EXERCISE] Cosine Similarity Explorer

1. Pick **three different queries** related to the student handbook (e.g., about attendance, grading, or health rules).
2. For each query, compute cosine similarity against **all 5 pages** of the handbook.
3. Does the highest-scoring page always contain the most relevant answer? Note any surprising results.


In [22]:
# Define three different queries related to the student handbook
queries_to_test = [
    "What is the protocol for sick students",
    "How can one be honor role?",
    "What is the policy on electronic devices?"
]

# Extract page content from docs_pdf for comparison
handbook_pages = [doc.page_content for doc in docs_pdf]

print("Calculating cosine similarity for each query against all handbook pages...")

for query_text in queries_to_test:
    print(f"\n--- Query: '{query_text}' ---")
    query_vec = encoder.encode([query_text])
    page_vecs = encoder.encode(handbook_pages)

    scores = cosine_similarity(query_vec, page_vecs)[0]

    # Pair scores with page index and sort by score
    ranked_pages = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    for rank, (page_idx, score) in enumerate(ranked_pages):
        print(f"  Rank {rank+1}: Page {page_idx + 1} (Score: {score:.4f})\n    Content Preview: {handbook_pages[page_idx][:100].replace('\n', ' ')}...")

Calculating cosine similarity for each query against all handbook pages...

--- Query: 'What is the protocol for sick students' ---
  Rank 1: Page 5 (Score: 0.4856)
    Content Preview: 4. OPERATIONS, DRESS CODE, & HEALTH Daily Standard Uniform Oakridge  Academy  relies  on  a  standar...
  Rank 2: Page 4 (Score: 0.3008)
    Content Preview: 3. STUDENT CODE OF CONDUCT Students represent Oakridge Academy at all times. Behavior that compromis...
  Rank 3: Page 1 (Score: 0.1759)
    Content Preview: OAKRIDGE ACADEMY Student & Parent Handbook Academic Year 2026-2027 Nurturing Excellence, Integrity, ...
  Rank 4: Page 3 (Score: 0.1713)
    Content Preview: 2. ACADEMIC POLICIES Oakridge Academy maintains a strict and rigorous curriculum standard. Student p...
  Rank 5: Page 2 (Score: 0.1677)
    Content Preview: 1. INTRODUCTION & GOVERNANCE Welcome from the Principal Welcome to Oakridge Academy. This handbook i...

--- Query: 'How can one be honor role?' ---
  Rank 1: Page 2 (Score: 0.2952)


No, the highest reanking page does not have the most relevant answer. The exact wording of the query can still influence the ranking, though less severely than with keyword search. Semantic similarity is better for synonyms but still benefits from clear and concise queries.

## Section 6. Chunking


In [23]:
# Set the loaded PDF as the document to by chunked
documents = docs_pdf

### 3.1 Fixed-Size

Traditional fixed-size chunking is the most basic way to split text, acting like a digital cookie cutter. It completely ignores both grammar and meaning, simply counting out a set number of characters or tokens (like 100 or 500) and chopping the text right there.

Here is how it works under the hood:

* **The Process:** You define a strict **chunk size** (e.g., 200 characters) and a **chunk overlap** (e.g., 50 characters) so consecutive chunks share some text to prevent total loss of context.
* **The Challenge:** Because it only counts characters, it frequently slices text right in the middle of a word, a sentence, or a crucial paragraph.
* **The Result:** It is incredibly fast and computationally cheap, but it often leaves LLMs with fragmented, confusing half-thoughts.

In [24]:
!pip install -q langchain-text-splitters

In [25]:
from langchain_text_splitters import CharacterTextSplitter

# Method 1: Fixed-Size Chunking

# Initialize splitter
text_splitter = CharacterTextSplitter(
  separator="\n\n",
  chunk_size=1000,
  chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

# Review the results
print(f"Split document into {len(chunks)} chunks.")

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())

Split document into 5 chunks.

--- Chunk 1 ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1

--- Chunk 2 ---
1. INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  our  community.  Our  goal  is  to  maintain  a  safe,  challenging,  and
supportive environment where every student can unlock their full academic and personal potential.
We expect students and parents to review this document together to ensure mutual compliance and a
shared vision for the upcoming academic year .
Mission Statement
Oakridge  Academy  is  dedicated  to  fostering  intellectual  curiosity,  critical  thinking,  and  moral
integrity. We prepare diverse student populations to become responsible global citi

### [EXERCISE] Vary the chunk size, chunk overlap, and separators

When configuring the splitter, you will interact with three main parameters:

- `chunk_size`: The maximum number of characters (or tokens) that a single chunk can contain.

- `chunk_overlap`: The number of characters that consecutive chunks share. Having an overlap creates a "sliding window" effect, ensuring that context at the boundary of a cut isn't completely lost.

- `separators`: The custom list of characters you want the splitter to recursively loop through.

#### Sub-Task 1: Modifying `chunk_size` and `chunk_overlap`

Modify the `chunk_size` to `300` and the `chunk_overlap` to `50`. Run the splitter again on the sample text and analyze how the boundaries change.

How many total chunks did you generate this time compared to before? Look closely at the overlapping regions—did this setup do a better job of keeping adjacent ideas connected?

In [26]:
# Add your code here!

text_splitter_1 = CharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks_1 = text_splitter_1.split_documents(docs_pdf)
print(f"Sub-Task 1: Split document into {len(chunks_1)} chunks.")

for i, chunk in enumerate(chunks_1):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())

Sub-Task 1: Split document into 5 chunks.

--- Chunk 1 ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1

--- Chunk 2 ---
1. INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  our  community.  Our  goal  is  to  maintain  a  safe,  challenging,  and
supportive environment where every student can unlock their full academic and personal potential.
We expect students and parents to review this document together to ensure mutual compliance and a
shared vision for the upcoming academic year .
Mission Statement
Oakridge  Academy  is  dedicated  to  fostering  intellectual  curiosity,  critical  thinking,  and  moral
integrity. We prepare diverse student populations to become responsible

#### Sub-Task 2: Breaking `chunk_size` and `chunk_overlap`

Now, intentionally break the configuration by setting your `chunk_size` to `100` and your `chunk_overlap` to `110`.

Run the code and observe the behavior. What happens conceptually and programmatically when your overlap is larger than your total chunk size? Why must `chunk_size` always be strictly greater than `chunk_overlap`?

In [27]:
# Add your code here!
try:
    text_splitter_2 = CharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=110
    )
    chunks_2 = text_splitter_2.split_documents(docs_pdf)
except ValueError as e:
    print(f"Sub-Task 2 Error: {e}")

Sub-Task 2 Error: Got a larger chunk overlap (110) than chunk size (100), should be smaller.


The overlap should be a subset of the chunk

#### Sub-Task 3: Varying Separators

Imagine you are chunking a chat transcript or a script where dialogue changes are marked by a dash (e.g., `"- Speaker A:"`).

Try passing a custom list to the `separators` parameter—such as `["\n\n", "\n-", " ", ""]`. Test it with a custom piece of text. How does adding a specific character sequence like `"\n-"` force the splitter to respect conversational boundaries instead of cutting mid-sentence?

In [28]:
# Add your code here!
dialogue_text = "Speaker A: Hello!\n- Speaker B: Hola Kamusta!\n- Speaker A: How are you?"
text_splitter_3 = CharacterTextSplitter(
    separator="\n-",
    chunk_size=50,
    chunk_overlap=0
)
chunks_3 = text_splitter_3.create_documents([dialogue_text])
print(f"Sub-Task 3: Split into {len(chunks_3)} chunks respecting dialogue.")
for i, chunk in enumerate(chunks_3):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())

Sub-Task 3: Split into 2 chunks respecting dialogue.

--- Chunk 1 ---
Speaker A: Hello!
- Speaker B: Hola Kamusta!

--- Chunk 2 ---
Speaker A: How are you?


We can try out recursively splitting text as well!

Instead of blindly cutting text at a hard character limit, the recursive splitter evaluates a prioritized list of separators—by default: `["\n\n", "\n", " ", ""]`.

Here is the step-by-step logic it executes:

1. **Paragraphs** (`\n\n`): It first attempts to split the text by double newlines. If the resulting paragraphs are smaller than your target chunk_size, it leaves them alone.

2. **Sentences** (`\n`): If a single paragraph is still larger than your chunk_size, the splitter moves to the next separator in the list and splits that paragraph into individual lines.

3. **Words** (` `): If a single line is still too large, it splits it by spaces into individual words.

4. **Characters** (`""`): As a last resort, if a single word exceeds the chunk size, it splits the word down to individual characters to strictly respect the limit.

By using this cascading logic, the splitter ensures that paragraphs and sentences are kept together for as long as mathematically possible, preserving the natural flow of information.

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000,
  chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

# Review the results
print(f"Split document into {len(chunks)} chunks.")

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())

Split document into 5 chunks.

--- Chunk 1 ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1

--- Chunk 2 ---
1. INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  our  community.  Our  goal  is  to  maintain  a  safe,  challenging,  and
supportive environment where every student can unlock their full academic and personal potential.
We expect students and parents to review this document together to ensure mutual compliance and a
shared vision for the upcoming academic year .
Mission Statement
Oakridge  Academy  is  dedicated  to  fostering  intellectual  curiosity,  critical  thinking,  and  moral
integrity. We prepare diverse student populations to become responsible global citi

### 3.2 Sentence-Based

To prepare text for AI models, we use sentence chunking to break large blocks of text into individual, grammatically correct sentences so information isn't awkwardly cut in half. To do this, we install NLTK (Natural Language Toolkit) and download its Punkt tokenizer, which is a pre-trained statistical model.

Here is exactly how it works under the hood:

- **The Challenge:** Computers don't inherently know if a period means the end of a sentence or just an abbreviation (like "Gen. Z" or "e.g.").

- **The Solution (Punkt):** Punkt looks at the words surrounding punctuation to smartly decide where a sentence actually ends.

- **The Result**: It cleanly slices your text into a clean list of complete sentences, ready for the next steps in your pipeline.

In [30]:
!pip install -q langchain langchain-community nltk

In [31]:
import nltk
from langchain_text_splitters import NLTKTextSplitter

# Download the required tokenization models (only needs to be run once)
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [32]:
# Initialize the NLTK splitter.
# chunk_size here represents the maximum characters per chunk, but it respects sentence boundaries.
text_splitter = NLTKTextSplitter(chunk_size=150, chunk_overlap=0)

chunks = text_splitter.split_documents(documents)

# Review the results
print(f"Split document into {len(chunks)} chunks.")

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())

Split document into 40 chunks.

--- Chunk 1 ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1

--- Chunk 2 ---
1.

INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy.

--- Chunk 3 ---
This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  our  community.

--- Chunk 4 ---
Our  goal  is  to  maintain  a  safe,  challenging,  and
supportive environment where every student can unlock their full academic and personal potential.

--- Chunk 5 ---
We expect students and parents to review this document together to ensure mutual compliance and a
shared vision for the upcoming academic year .

--- Chunk 6 ---
Mission Statement
Oakridge  Academy  is  dedicated  to  fostering  intellectual  curiosity,  critical  thinking,  and  moral
integrity.

--

### [EXERCISE] Vary the chunk size, chunk overlap, and separators

Similar to the previous exercise, try varying the chunk size, chunk overlap, and separators

#### Sub-Task 1: Testing the Abbreviations Trap
Compare how the `NLTKTextSplitter` handles the words `Dr.`, `a.m.`, and `Corp.` versus how a standard character splitter would handle them if you set a very small `chunk_size` (e.g., `40`).

Does the NLTK splitter safely keep `"Dr. Smith arrived at the lab at 8:00 a.m."` as a single, unbroken sentence, or does it slice it at the abbreviations? Why is this important for data integrity?

In [33]:
# Add your code here!
nltk_splitter_1 = NLTKTextSplitter(chunk_size=40, chunk_overlap=0)
nltk_chunks_1 = nltk_splitter_1.create_documents("Dr. Smith arrived at the lab at 8:00 a.m.")
for i, chunk in enumerate(nltk_chunks_1):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())


--- Chunk 1 ---
D

--- Chunk 2 ---
r

--- Chunk 3 ---
.

--- Chunk 4 ---
S

--- Chunk 5 ---
m

--- Chunk 6 ---
i

--- Chunk 7 ---
t

--- Chunk 8 ---
h

--- Chunk 9 ---
a

--- Chunk 10 ---
r

--- Chunk 11 ---
r

--- Chunk 12 ---
i

--- Chunk 13 ---
v

--- Chunk 14 ---
e

--- Chunk 15 ---
d

--- Chunk 16 ---
a

--- Chunk 17 ---
t

--- Chunk 18 ---
t

--- Chunk 19 ---
h

--- Chunk 20 ---
e

--- Chunk 21 ---
l

--- Chunk 22 ---
a

--- Chunk 23 ---
b

--- Chunk 24 ---
a

--- Chunk 25 ---
t

--- Chunk 26 ---
8

--- Chunk 27 ---
:

--- Chunk 28 ---
0

--- Chunk 29 ---
0

--- Chunk 30 ---
a

--- Chunk 31 ---
.

--- Chunk 32 ---
m

--- Chunk 33 ---
.


#### Sub-Task 2: Finding the Sentence Ceiling
Keep your `chunk_size` at `120`, but add a massive, continuous sentence to the `tricky_text` string that is well over 200 characters long (e.g., a long, rambling run-on sentence). Run the splitter.

What happens when an individual sentence is inherently larger than your configured `chunk_size`? Does NLTK force a cut mid-sentence, or does it expand the chunk size dynamically to preserve the sentence?

In [34]:
# Add your code here!
nltk_splitter_2 = NLTKTextSplitter(chunk_size=120, chunk_overlap=0)
nltk_chunks_2 = nltk_splitter_2.create_documents("Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.")
for i, chunk in enumerate(nltk_chunks_2):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())


--- Chunk 1 ---
D

--- Chunk 2 ---
r

--- Chunk 3 ---
.

--- Chunk 4 ---
S

--- Chunk 5 ---
m

--- Chunk 6 ---
i

--- Chunk 7 ---
t

--- Chunk 8 ---
h

--- Chunk 9 ---
a

--- Chunk 10 ---
r

--- Chunk 11 ---
r

--- Chunk 12 ---
i

--- Chunk 13 ---
v

--- Chunk 14 ---
e

--- Chunk 15 ---
d

--- Chunk 16 ---
a

--- Chunk 17 ---
t

--- Chunk 18 ---
t

--- Chunk 19 ---
h

--- Chunk 20 ---
e

--- Chunk 21 ---
l

--- Chunk 22 ---
a

--- Chunk 23 ---
b

--- Chunk 24 ---
a

--- Chunk 25 ---
t

--- Chunk 26 ---
8

--- Chunk 27 ---
:

--- Chunk 28 ---
0

--- Chunk 29 ---
0

--- Chunk 30 ---
a

--- Chunk 31 ---
.

--- Chunk 32 ---
m

--- Chunk 33 ---
.

--- Chunk 34 ---
D

--- Chunk 35 ---
r

--- Chunk 36 ---
.

--- Chunk 37 ---
S

--- Chunk 38 ---
m

--- Chunk 39 ---
i

--- Chunk 40 ---
t

--- Chunk 41 ---
h

--- Chunk 42 ---
a

--- Chunk 43 ---
r

--- Chunk 44 ---
r

--- Chunk 45 ---
i

--- Chunk 46 ---
v

--- Chunk 47 ---
e

--- Chunk 48 ---
d

--- Chunk 49 ---
a

--- Chunk 50 ---
t

--- Chun

#### Sub-Task 3: Adding Overlap to Sentences
Modify your `NLTKTextSplitter` configuration to include a `chunk_overlap` of `30`.

Since NLTK operates on sentence boundaries, how does it handle character overlap? Does it pull the last few *words* of the previous sentence into the next chunk, or does it grab the entire preceding *sentence*? Examine the output closely.

In [35]:
# Add your code here!
# Add your code here!
nltk_splitter_3 = NLTKTextSplitter(chunk_size=120, chunk_overlap=30)
nltk_chunks_3 = nltk_splitter_3.create_documents("Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.Dr. Smith arrived at the lab at 8:00 a.m.")
for i, chunk in enumerate(nltk_chunks_3):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content.strip())


--- Chunk 1 ---
D

--- Chunk 2 ---
r

--- Chunk 3 ---
.

--- Chunk 4 ---
S

--- Chunk 5 ---
m

--- Chunk 6 ---
i

--- Chunk 7 ---
t

--- Chunk 8 ---
h

--- Chunk 9 ---
a

--- Chunk 10 ---
r

--- Chunk 11 ---
r

--- Chunk 12 ---
i

--- Chunk 13 ---
v

--- Chunk 14 ---
e

--- Chunk 15 ---
d

--- Chunk 16 ---
a

--- Chunk 17 ---
t

--- Chunk 18 ---
t

--- Chunk 19 ---
h

--- Chunk 20 ---
e

--- Chunk 21 ---
l

--- Chunk 22 ---
a

--- Chunk 23 ---
b

--- Chunk 24 ---
a

--- Chunk 25 ---
t

--- Chunk 26 ---
8

--- Chunk 27 ---
:

--- Chunk 28 ---
0

--- Chunk 29 ---
0

--- Chunk 30 ---
a

--- Chunk 31 ---
.

--- Chunk 32 ---
m

--- Chunk 33 ---
.

--- Chunk 34 ---
D

--- Chunk 35 ---
r

--- Chunk 36 ---
.

--- Chunk 37 ---
S

--- Chunk 38 ---
m

--- Chunk 39 ---
i

--- Chunk 40 ---
t

--- Chunk 41 ---
h

--- Chunk 42 ---
a

--- Chunk 43 ---
r

--- Chunk 44 ---
r

--- Chunk 45 ---
i

--- Chunk 46 ---
v

--- Chunk 47 ---
e

--- Chunk 48 ---
d

--- Chunk 49 ---
a

--- Chunk 50 ---
t

--- Chun

### 3.3 Semantic Chunking

Semantic chunking groups text based on its actual meaning rather than rigid rules or punctuation. By using an embedding model to turn sentences into mathematical vectors, it calculates how conceptually similar they are and groups related ideas together.

Here is how it works under the hood:

- **The Challenge:** Traditional chunkers might cut a paragraph in half mid-thought, destroying the context an LLM needs to understand it.

- **The Solution (Embeddings):** Semantic chunking analyzes the sentences and measures the conceptual "distance" between them.

- **The Result:** It draws a boundary only when the topic completely shifts, keeping relevant information grouped together in cohesive chunks.

First, we must set up an embeddings model.

In [36]:
# Pull the embedding model (nomic-embed-text is highly recommended and lightweight)
print("📥 Downloading the embedding model...")
!ollama pull nomic-embed-text

print("🎉 Ollama is ready to go!")

📥 Downloading the embedding model...

🎉 Ollama is ready to go!


In [37]:
!pip install -q langchain langchain-ollama langchain-experimental

In [38]:
from langchain_ollama import OllamaEmbeddings

# Initialize the free Ollama embeddings pointing to your local Colab instance
embeddings = OllamaEmbeddings(model="nomic-embed-text")

Then, we can set up the text splitter!

In [39]:
from langchain_experimental.text_splitter import SemanticChunker

# Setup the semantic chunker
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

# Split the text
chunks = text_splitter.split_documents(documents)

# Review the results
print(f"Split document into {len(chunks)} chunks.")

/tmp/ipykernel_4889/414119074.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Split document into 9 chunks.


### [EXERCISE] Vary the `breakpoint_threshold_type`

There are 3 commonly used Breakpoint Threshold Types:
- **Percentile:** Splits text based on a percentile of distance drops between sentences (e.g., any drop in the top 10%).
- **Standard Deviation:** Splits text when a distance drop exceeds a specified number of standard deviations from the mean.
- **Interquartile Range (IQR):** Splits text based on the statistical interquartile range, making it highly robust against extreme outliers.

You can also try varying the `breakpoint_threshold_amount` — the specific numerical value (the percentile rank, number of standard deviations, IQR multiplier, or raw gradient threshold) used to trigger a split.

In [40]:
# Add your code here!
# percentile
semantic_chunker_percentile = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=10
)
percentile_chunks = semantic_chunker_percentile.split_documents(docs_pdf)
print(f"Generated {len(percentile_chunks)} chunks using percentile.")
# std dev
semantic_chunker_std = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1.5
)
std_chunks = semantic_chunker_std.split_documents(docs_pdf)
print(f"Generated {len(std_chunks)} chunks using standard_deviation.")

# Using interquartile
semantic_chunker_iqr = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="interquartile",
    breakpoint_threshold_amount=1.0
)
iqr_chunks = semantic_chunker_iqr.split_documents(docs_pdf)
print(f"Generated {len(iqr_chunks)} chunks using IQR.")

Generated 39 chunks using percentile.
Generated 11 chunks using standard_deviation.
Generated 11 chunks using IQR.


### [EXERCISE] Exploring Advanced LangChain Splitters

Different data formats require different splitting strategies. LangChain provides specialized splitters that understand the structural rules of specific file types.

1. Open the [LangChain Text Splitters Reference API](https://reference.langchain.com/python/langchain-text-splitters).
2. Choose **one** specialized splitter to experiment with, such as:
   * `MarkdownTextSplitter` (splits naturally by markdown headers)
   * `PythonCodeTextSplitter` (respects code block boundaries and syntax)
   * `TokenTextSplitter` (chunks by exact LLM token counts rather than characters)
3. Write a code cell below to initialize your chosen splitter, test it on a custom sample text, and print the output chunks.

How does your chosen splitter preserve document context better than a standard character splitter?

In [41]:
# Add your code here!
from langchain_text_splitters import PythonCodeTextSplitter

python_code = """
def say_hello(name="World"):
    # This function prints a greeting
    message = f"Hello, {name}!"
    print(message)
    return message

def main():
    default_greeting = say_hello()
    user_greeting = say_hello("Marieeeel")

if __name__ == "__main__":
    main()
"""

splitter_python = PythonCodeTextSplitter(
    chunk_size=100,
    chunk_overlap=0
)

chunks_python = splitter_python.create_documents([python_code])

print(f"Generated {len(chunks_python)} chunks.\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print("-" * 20)

Generated 4 chunks.

--- Chunk 1 ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1
--------------------
--- Chunk 2 ---
1. INTRODUCTION & GOVERNANCE
Welcome from the Principal
Welcome to Oakridge Academy. This handbook is designed to outline the expectations, policies, and
responsibilities  that  govern  our  community. Our  goal  is  to  maintain  a  safe,  challenging,  and
supportive environment where every student can unlock their full academic and personal potential.
--------------------
--- Chunk 3 ---
We expect students and parents to review this document together to ensure mutual compliance and a
shared vision for the upcoming academic year . Mission Statement
Oakridge  Academy  is  dedicated  to  fostering  intellectual  curiosity,  critical  thinking,  and  moral
integrity. We prepare diverse student

## Section 7. Embedding Models

This embedding step simply takes that list and translates each text chunk into a dense numerical vector that captures its core concepts, preparing the data for the AI to search through later.

This is exactly the same way the embeddings model in semantic chunking was set up.

In [42]:
# Pull the embedding model (nomic-embed-text is highly recommended and lightweight)
print("📥 Downloading the embedding model...")
!ollama pull nomic-embed-text

print("🎉 Ollama is ready to go!")

📥 Downloading the embedding model...

🎉 Ollama is ready to go!


In [43]:
from langchain_community.embeddings import OllamaEmbeddings

# Initialize the embedding model (Ensure your Ollama background server is running!)
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_4889/701672538.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings_model = OllamaEmbeddings(model="nomic-embed-text")


If you want to generate the vectors separately, you can run the following:

In [44]:
# Send the list to Ollama to generate the mathematical vectors
print(f"Generating embeddings for {len(chunks)} chunks...")
vectors = embeddings_model.embed_documents(chunks)

# Inspect the results
print("✨ Embeddings successfully generated!")
print(f"Total vectors created: {len(vectors)}")
print(f"Dimensions per vector: {len(vectors[0])}")

Generating embeddings for 9 chunks...
✨ Embeddings successfully generated!
Total vectors created: 9
Dimensions per vector: 768


### [EXERCISE] Try using a different open-source embedding model (e.g., BGE-M3, E5-Large) and compare their performance trade-offs (e.g., latency, size, MTEB score) for the FAQ bot knowledge base.

In [45]:
# Add your code here!

import time
from sentence_transformers import SentenceTransformer

model_bge = SentenceTransformer("BAAI/bge-m3")
model_e5 = SentenceTransformer("intfloat/e5-large-v2")

faq_queries = [
    "How do I apply for a Leave of Absence?",
    "What are the honor role requirements?",
    "What is the vision and mission of the?"
]

start_time = time.time()
embeddings_bge = model_bge.encode(faq_queries)
bge_latency = time.time() - start_time

e5_queries = [f"query: {q}" for q in faq_queries]

start_time = time.time()
embeddings_e5 = model_e5.encode(e5_queries)
e5_latency = time.time() - start_time

print(f"BGE-M3 Dimensions: {embeddings_bge.shape[1]} | Latency: {bge_latency:.4f}s")
print(f"E5-Large Dimensions: {embeddings_e5.shape[1]} | Latency: {e5_latency:.4f}s")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

BGE-M3 Dimensions: 1024 | Latency: 8.5809s
E5-Large Dimensions: 1024 | Latency: 1.0545s


## Section 8. Vector Storage

Now that we have chopped our raw text into distinct chunks, we need a way to store them so an LLM can easily search through them later. This is where ChromaDB and our Ollama embedding model work together as a team.

Here is exactly how this process unfolds under the hood:

- **The Indexing Process (Chroma.from_texts):** When we pass our list of chunks to Chroma alongside our embeddings_model, we aren't just saving plain text. Chroma automatically sends each text chunk to Ollama, receives its unique mathematical vector, and pairs them up in the database.

- **The Semantic Filing Cabinet:** Think of ChromaDB as a giant, multi-dimensional map. It uses those vectors from Ollama to physically plot your chunks in a "meaning space." Chunks about similar topics are placed right next to each other, while completely unrelated topics are placed far apart.

- **The Retrieval Magic (similarity_search):** When you ask the database a question, Chroma uses Ollama one last time to turn your query into a vector. It then looks at its map, finds the chunks closest to your question's mathematical location, and pulls them instantly—even if your question didn't use the exact same words as the text.

In [47]:
!pip install opentelemetry-api opentelemetry-sdk langchain-chroma

In [13]:
# 1. Force uninstall the conflicting packages to clear the cache
!pip uninstall -y opentelemetry-api opentelemetry-sdk langchain-chroma

# 2. Reinstall them cleanly so they grab the latest, matching versions
!pip install opentelemetry-api opentelemetry-sdk langchain-chroma

Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: langchain-chroma 1.1.0
Uninstalling langchain-chroma-1.1.0:
  Successfully uninstalled langchain-chroma-1.1.0
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached langchain_chroma-1.1.0-py3-none-any.whl (12 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depende

In [48]:
from langchain_chroma import Chroma

# Create an in-memory Chroma database populated with your text and embedding model
print("Creating Chroma vector database and indexing chunks...")
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model
)

print("🚀 Chroma database is ready and fully indexed!")

Creating Chroma vector database and indexing chunks...
🚀 Chroma database is ready and fully indexed!


## Section 9. Advanced Information Retrieval

### 6.1 Similarity Search
Once text is inside our vector database, we retrieve information using Similarity Search.
- The Concept: Traditional databases look for exact word matches (keyword search). Vector databases look for conceptual matches. ChromaDB takes your search phrase, turns it into a vector via Ollama, and calculates the mathematical "distance" between your query and your stored chunks.
- Chroma's Distance Metric: By default, Chroma calculates squared Euclidean distance ($L^2$ distance). Lower numbers mean the documents are closer together and more similar, while higher numbers mean they are different.

While fetching the closest chunks works fine, a big issue in production RAG systems is redundancy. If your document repeats the same fact three times across different paragraphs, a basic Top-K search will return all three identical paragraphs, wasting your budget and cluttering the AI's memory.

To fix this, we use two different selection strategies:

- Top-K Selection (Pure Relevance): It ruthlessly picks the absolute closest items to your query, even if they say the exact same thing.

- Maximum Marginal Relevance (MMR) (Relevance + Diversity): MMR is a balancing act. It initially looks at a wide pool of candidate chunks (fetch_k). It grabs the closest match first, but before picking the second or third match, it checks how similar they are to the chunks it already picked. If a chunk is redundant, MMR penalizes it and looks for a different, distinct piece of information.

In [49]:
query = "What is the university dress code?"

#### 6.2.1 Top-K Selection

In [50]:
# Approach 1: Pure Top-K (High risk of repetitive content)
print("Fetching pure Top-K results...")
top_k_results = db.similarity_search(query, k=3)

print(f"Found {len(top_k_results)} matching chunks:\n")
for i, doc in enumerate(top_k_results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)

Fetching pure Top-K results...
Found 3 matching chunks:

--- Chunk 1 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours. Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or institutional plaid pleated skirts, solid white collared dress shirt, and dark leather
dress shoes. Fridays (Spirit Days): Approved Oakridge polo shirts paired with standard neat denim jeans
(free from distressing, holes, or visible frayed patches). Campus Health & Medical Administration
The health suite is continuously staffed by a registered nurse. Students requiring medical updates or
medication administration during active operating hours must strictly adhere to the infrastructure
outlined below:
Medication Class Storage O

If you want to see the scores, use `similarity_search_with_score`.

In [51]:
# Run search with distance scores (Seeing the math)
# This returns a list of tuples: (Document object, distance_score)
results_with_scores = db.similarity_search_with_score(query, k=3)

print("\n--- Top Matching Results with Scores ---")
for doc, score in results_with_scores:
    print(f"\n[Distance Score: {score:.4f}] (Lower = More Similar)")
    print(f"Content: {doc.page_content[:150]}...")


--- Top Matching Results with Scores ---

[Distance Score: 335.0265] (Lower = More Similar)
Content: 4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  ...

[Distance Score: 401.3218] (Lower = More Similar)
Content: Students are expected to transition through routes silently,
swiftly, and in precise configurations under structural supervision from their respective...

[Distance Score: 420.4449] (Lower = More Similar)
Content: 3. STUDENT CODE OF CONDUCT
Students represent Oakridge Academy at all times. Behavior that compromises the dignity, emotional
peace, or safety of our ...


#### 6.2.2 Maximum Marginal Relevance (MMR)

In [52]:
# Approach 2: Maximal Marginal Relevance (Diverse results)
print("Fetching MMR results for better diversity...")
mmr_results = db.max_marginal_relevance_search(
    query,
    k=3,          # The final number of chunks to return to the LLM
    fetch_k=10,   # Look at the top 10 closest chunks initially before filtering for diversity
    lambda_mult=0.5 # Balance factor (1.0 = pure relevance, 0.0 = pure diversity)
)

print(f"Found {len(mmr_results)} matching chunks:\n")
for i, doc in enumerate(mmr_results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)

Fetching MMR results for better diversity...
Found 3 matching chunks:

--- Chunk 1 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours. Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or institutional plaid pleated skirts, solid white collared dress shirt, and dark leather
dress shoes. Fridays (Spirit Days): Approved Oakridge polo shirts paired with standard neat denim jeans
(free from distressing, holes, or visible frayed patches). Campus Health & Medical Administration
The health suite is continuously staffed by a registered nurse. Students requiring medical updates or
medication administration during active operating hours must strictly adhere to the infrastructure
outlined below:
Medication C

### 6.3 Context Window Retrieval

The **Context Window** strategy, discussed in Part 1, is an alternative to Top-K. Instead of returning only the
highest-similarity chunk, it also returns the **neighboring chunks** (the chunks immediately before and after
the best match). This is useful when:

- Related information is spread across adjacent paragraphs (e.g., a multi-step policy).
- Your documents have a clear structure like guides, lessons, or directories.
- A small chunk size means individual chunks lose context on their own.


In [53]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity  # Assuming you are using sklearn for this

# Re-split with small chunks so neighbours are meaningful
small_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
small_chunks   = small_splitter.split_documents(docs_pdf)
small_texts    = [c.page_content for c in small_chunks]

window_query = "What are the rules around uniforms on Fridays?"

# Use the 'embeddings_model' variable with the proper LangChain methods
print("Embedding chunks and query...")
all_vecs   = embeddings_model.embed_documents(small_texts)
query_vec2 = [embeddings_model.embed_query(window_query)]

# Calculate similarities
sims        = cosine_similarity(query_vec2, all_vecs)[0]
best_idx    = int(np.argmax(sims))

window_size = 1  # how many neighbours to grab on each side
start = max(0, best_idx - window_size)
end   = min(len(small_texts) - 1, best_idx + window_size)

print(f"\nQuery: {window_query}")
print(f"Best matching chunk index: {best_idx} (similarity: {sims[best_idx]:.4f})")
print(f"Returning chunks {start} → {end} (context window = ±{window_size})\n")

for i in range(start, end + 1):
    label = " ← BEST MATCH" if i == best_idx else ""
    print(f"--- Chunk {i}{label} ---")
    print(small_texts[i].strip())
    print()

Embedding chunks and query...

Query: What are the rules around uniforms on Fridays?
Best matching chunk index: 20 (similarity: 0.6362)
Returning chunks 19 → 21 (context window = ±1)

--- Chunk 19 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours.

--- Chunk 20 ← BEST MATCH ---
Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or institutional plaid pleated skirts, solid white collared dress shirt, and dark leather
dress shoes.
Fridays (Spirit Days): Approved Oakridge polo shirts paired with standard neat denim jeans

--- Chunk 21 ---
(free from distressing, holes, or visible frayed patches).
Campus Health & Medical Administration
The health suite is continuously staffed by a registered nurse. St

### [EXERCISE] Tune the Window Size

Change `window_size` to `2` and then `3`. Answer the following:
1. At what window size does the context become helpful vs. noisy?
2. When would you prefer Context Window retrieval over MMR? Give an example use case.


In [54]:
# Add your code here!
print("Window size == 2")
window_size = 2

start = max(0, best_idx - window_size)
end   = min(len(small_texts) - 1, best_idx + window_size)

print(f"\nQuery: {window_query}")
print(f"Best matching chunk index: {best_idx} (similarity: {sims[best_idx]:.4f})")
print(f"Returning chunks {start} → {end} (context window = ±{window_size})\n")

for i in range(start, end + 1):
    label = " ← BEST MATCH" if i == best_idx else ""
    print(f"--- Chunk {i}{label} ---")
    print(small_texts[i].strip())
    print()

print("Window size == 3")
window_size = 3

start = max(0, best_idx - window_size)
end   = min(len(small_texts) - 1, best_idx + window_size)

print(f"\nQuery: {window_query}")
print(f"Best matching chunk index: {best_idx} (similarity: {sims[best_idx]:.4f})")
print(f"Returning chunks {start} → {end} (context window = ±{window_size})\n")

for i in range(start, end + 1):
    label = " ← BEST MATCH" if i == best_idx else ""
    print(f"--- Chunk {i}{label} ---")
    print(small_texts[i].strip())
    print()

Window size == 2

Query: What are the rules around uniforms on Fridays?
Best matching chunk index: 20 (similarity: 0.6362)
Returning chunks 18 → 22 (context window = ±2)

--- Chunk 18 ---
Oakridge Academy Student Handbook 2026-2027 Page 4

--- Chunk 19 ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours.

--- Chunk 20 ← BEST MATCH ---
Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or institutional plaid pleated skirts, solid white collared dress shirt, and dark leather
dress shoes.
Fridays (Spirit Days): Approved Oakridge polo shirts paired with standard neat denim jeans

--- Chunk 21 ---
(free from distressing, holes, or visible frayed patches).
Campus Health & Medical Administration
The health

Context Window retrieval would be preferred over MMR when you have documents with a sequential structure, where the meaning of a relevant chunk is heavily dependent on its neighbors. For example, in a step-by-step guide, the full context might only be understood by reading the preceding and succeeding sections.

MMR, on the other hand, is better when you need answers to a query from a collection of documents that might have overlapping but not strictly sequential information. For instance, if you're searching a large corpus of science articles for different perspectives on a topic, MMR can provide a wider variety of information.

## Section 10. Generation

Now that our data is chunked and stored, it is time to look at the **Generation** stage of RAG. How exactly does adding context change an LLM's response?

To understand this, we will run a comparative experiment testing the exact same question against **three distinct architectures**:
1. **No Context:** The model relies purely on its pre-trained internal weights.
2. **Full Context (Context Stuffing):** We paste the entire un-chunked document directly into the prompt.
3. **Retrieval-Augmented Generation (RAG):** We use a vector database to search, retrieve only the most relevant *semantic chunks*, and pass those to the model.

First, let's set up the LLM and the query

In [55]:
!ollama pull gemma3:1b

# Or specify other models, e.g.
# # !ollama pull smollm2

In [56]:
query = "What is the dress code for major exams according to the handbook?"

### No Context

Similar to earlier, let's start with the No Context approach.

The No Context approach relies entirely on the pre-trained internal memory of the LLM. As discussed earlier, the model answers questions based solely on data it digested up until its training cutoff.

This method is incredibly fast and inexpensive because the prompt payload is tiny. However, it is highly prone to confident hallucinations or flat refusals when asked about real-time events, private data, or niche corporate knowledge.

In [57]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm = Ollama(model="gemma3:1b")

# 1. Reuse the exact same query and LLM instance from before
naive_query = query

# 2. Define a simple prompt that asks the question directly
prompt_template = ChatPromptTemplate.from_template("""
You are an AI assistant. Answer the user's question to the best of your ability.

USER QUESTION:
{question}

ANSWER:
""")

print(f"Sending the query directly to the LLM WITHOUT any database context...\n")

# 3. Set up the direct chain
naive_chain = prompt_template | llm | StrOutputParser()

# 4. Execute the chain
naive_response = naive_chain.invoke({
    "question": naive_query
})

print("--- Final AI Response (WITHOUT Context) ---")
print(naive_response)

Sending the query directly to the LLM WITHOUT any database context...

--- Final AI Response (WITHOUT Context) ---
Please provide me with the “handbook” you’re referring too! I need access to that document, or a link to it, in order to answer your question about the dress code for exams. 😊

Once you paste (or link) the relevant document, I will be able to give you accurate information.


### Full Context
The Full Context method, often called "context stuffing," bypasses databases entirely by pasting raw, un-chunked source documents directly into the prompt alongside the user's question.

Expect near-perfect factual accuracy for shorter files. However, as the document grows and starts pushing past the expected context window, the model's attention degrades rapidly. You will notice a sharp spike in hallucinations, missed details ("lost in the middle"), or outright model failure as it overflows its memory capacity.

Another downside is a major bottleneck in production: soaring token costs, high latency, and hard context window crashes when files scale up.

In [65]:
# 1. Reuse the exact same query and LLM instance
full_context_query = query

# 2. Re-read or reference your completely un-chunked, raw original text
raw_document_context = "\n".join([doc.page_content for doc in docs_pdf])

# 3. Reuse our strict RAG prompt template from Step 7
print("Sending the ENTIRE document to the LLM at once (This may take longer)...")

# Execute the same pipeline
full_context_chain = prompt_template | llm | StrOutputParser()

import time
start_time = time.time()

full_context_response = full_context_chain.invoke({
    "context": raw_document_context,
    "question": full_context_query
})

end_time = time.time()
print(f"Done! Generation took {end_time - start_time:.2f} seconds.")

print("\n--- Final AI Response (With FULL Document) ---")
print(full_context_response)

Sending the ENTIRE document to the LLM at once (This may take longer)...
Done! Generation took 92.83 seconds.

--- Final AI Response (With FULL Document) ---
The dress code for major exams, outlined in the Oakridge Academy Student Handbook, specifies that students must present themselves in “navy blue blazer featuring the official school crest” during Mondays through Thursdays and “Approved Oak Ridge Polo Shirts paired with standard neat denim jeans (free from distressing, holes, or visible frayed patches)” on Fridays.


### Using RAG

The RAG architecture balances accuracy and efficiency. It converts documents into searchable semantic chunks, queries a vector database to extract only the most relevant snippets, and feeds just those specific pieces to the LLM.

This is the production standard for large datasets. It keeps costs and latency low while maintaining accuracy. However, its success depends entirely on your chunking strategy; poor retrieval will directly cause the model to generate incomplete or wrong answers.

In [62]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize our local generative model via Ollama
print("Initializing Ollama LLM for generation...")
llm = Ollama(model="gemma3:1b")

# 2. Define the prompt template that stitches the context and question together
# This template acts as a strict guardrail for the AI's behavior.
prompt_template = ChatPromptTemplate.from_template("""
You are a helpful, factual AI assistant. Your task is to answer the user's question using ONLY the provided context.
If the context does not contain the answer, cleanly state that you do not know. Do not make up answers.

---
RETRIEVED CONTEXT FROM DATABASE:
{context}

---
USER QUESTION:
{question}

ANSWER:
""")

# 3. Pull the EXACT query and output from our previous retrieval cell
# We extract the text content from each LangChain document and join them together.
user_query = query
retrieved_context = "\n\n".join([doc.page_content for doc in mmr_results])

# 4. Format the prompt and send it to the LLM
print(f"\nSending context and query to the LLM...\n")

# Chain components together using LangChain Expression Language (LCEL)
rag_chain = prompt_template | llm | StrOutputParser()

# Execute the chain
response = rag_chain.invoke({
    "context": retrieved_context,
    "question": user_query
})

print("--- Query ---")
print(user_query)

print("--- Context ---")
print(retrieved_context)

print("--- Final AI Response ---")
print(response)

Initializing Ollama LLM for generation...

Sending context and query to the LLM...

--- Query ---
What is the dress code for major exams according to the handbook?
--- Context ---
4. OPERATIONS, DRESS CODE, & HEALTH
Daily Standard Uniform
Oakridge  Academy  relies  on  a  standard  uniform  to  minimize  social  divisions  and  promote
institutional unity. All students must present themselves in clean, properly sized school attire during
regular operating hours. Mondays through Thursdays: Navy blue blazer featuring the official school crest, tailored khaki
trousers or institutional plaid pleated skirts, solid white collared dress shirt, and dark leather
dress shoes. Fridays (Spirit Days): Approved Oakridge polo shirts paired with standard neat denim jeans
(free from distressing, holes, or visible frayed patches). Campus Health & Medical Administration
The health suite is continuously staffed by a registered nurse. Students requiring medical updates or
medication administration during a

### [EXERCISE] Try other queries and compare the output with full context, with RAG, and without context.

Test different types of questions across all three methods. Fill out the matrix below in your notes to evaluate the trade-offs:

| Metric | No Context | Full Context | RAG |
| :--- | :--- | :--- | :--- |
| **Speed / Latency** | Fast ⚡ | Slow 🐢 | Moderate ⚖️ |
| **Hallucination Risk** | High | Low | Low-Medium (Depends on chunking) |
| **Handling Outdated Info**| Poor | Excellent | Excellent |
| **System Complexity** | None | Low | High (Requires DB & Embeddings) |

In [66]:
# Add your code here!

test_query = "What happens if I miss an exam because of a family emergency?"

# 1. RAG Context
rag_results = db.similarity_search(test_query, k=2)
rag_context = "\n".join([doc.page_content for doc in rag_results])
print("--- RAG Answer ---")
response = rag_chain.invoke({
    "context": retrieved_context,
    "question": test_query
})
print(response)

# 2. Full Context
full_context_text = "\n".join([doc.page_content for doc in docs_pdf])
print("\n--- Full Context Answer ---")
full_context_response = full_context_chain.invoke({
    "context": raw_document_context,
    "question": test_query
})
print(full_context_response)

# 3. No Context
print("\n--- No Context Answer ---")
naive_response = naive_chain.invoke({
    "question": test_query
})
print(naive_response)

--- RAG Answer ---
The context provided does not state what happens when you miss an exam due to a family emergency.

--- Full Context Answer ---
If students miss an examination due to a family emergency, the following consequences may occur, as outlined in the academic policy:

Students whose cumulative grade point average falls below a 2.00 at any formal marking period will be placed on Academic Probation. Academic Probation forces an automatic intervention sequence including mandatory bi-weekly progress updates, targeted faculty tutoring sessions, and temporary suspension from all extracurricular activities, including interscholistic athletics.

--- No Context Answer ---
Okay, here’s some potential steps and information regarding what might happen when you miss an examination due to a family emergency. It sounds very challenging - handling this situation can be tough, so please remember that it's perfectly normal to feel stressed. Here are several options available to you:

**1. Ass

## Section 11. University FAQ bot

Prototype a **University FAQ Bot** that answers student questions from the student handbook

**What we want**:

- Semantic Context Matching: The system must locate the correct policy chunk even if the student uses casual language (e.g., matching "flunking out" to "Academic Dismissal").

- Strict Grounding Guardrails: The LLM must rely only on the provided handbook text to prevent hallucination. If a policy is missing from the source text, it must explicitly state its lack of information rather than guessing.

- Source Attribution: The output must cite the exact document section and page number so the student can verify the official rule.


Sample user input:
```
Hey, I had a family emergency and completely missed my physics midterm yesterday. Am I going to get a zero or can I take a makeup test?
```

Sample Grounding Sources:
```
[Source: Student_Handbook_2026.pdf | Section 4.2: Examination Absences | Page 14]
"Absences from scheduled midterms or final examinations are classified as either Excused or Unexcused. Unexcused absences automatically result in a grade of 0.0 for the respective evaluation. Excused absences are granted only under extreme, documented circumstances (e.g., medical emergencies, severe family crises). To secure an excused absence, the student must submit an Official Absence Petition alongside verifiable documentation to the Office of Student Affairs within forty-eight (48) hours of the missed examination period. Upon administrative approval, the student is permitted to sit for a rescheduled makeup examination, which must be completed within 7 calendar days of the original test date."

[Source: Student_Handbook_2026.pdf | Section 4.5: Academic Grievances | Page 16]
"Students wishing to appeal an academic grade or an unexcused exam absence penalty must initiate a formal appeal with the Department Chair no later than five (5) business days following the posting of the contested grade."
```

Sample LLM Output:
```
Based on the Student Handbook (Section 4.2, Page 14), you will not automatically receive a zero, provided you submit official documentation.

Here are the specific steps you must follow:
1. You must submit an Official Absence Petition along with verified documentation of your family emergency to the Office of Student Affairs.
2. This petition must be filed within forty-eight (48) hours of the missed exam.
3. If approved, you will be granted a excused absence, and you must coordinate with your physics professor to take a rescheduled makeup exam within 7 calendar days.

If you fail to submit the petition within the 48-hour window, the absence will remain unexcused, and a grade of 0.0 will be recorded for that midterm.

```

### [EXERCISE] Improving the Student Handbook RAG Pipeline

In this exercise, you will act as an AI QA Engineer. Your goal is to tweak, break, and optimize the **Retrieval-Augmented Generation (RAG)** pipeline you just built for the University Student Handbook FAQ Bot.

Instead of treating the pipeline as a "black box," you will modify parameters at every stage of the lifecycle to observe how your architectural choices change the accuracy, cost, and safety of the final answers.

---

### **The RAG Pipeline Checklist**

#### **1. Ingest**
- [X] **Loaders:** Toggle between loading data from a static document (`PyPDFLoader`) versus a dynamic web page (`WebBaseLoader`). Observe if formatting changes affect text extraction.

#### **2. Chunk**
- [X] **Chunk Size & Overlap:** Experiment with splitting your text into very small pieces (e.g., 200 characters) versus massive pieces (e.g., 2,000 characters).

#### **3. Embed**
- [X] **Dimensions & Latency:** Compare a local open-source model against a cloud API model. Track how fast they convert your text into math coordinates.

#### **4. Store**
- [X] **DB Initialization:** Ensure your database is correctly caching your vectors locally on disk so you don't re-embed data on every notebook run.

#### **5. Retrieve**
- [X] **Search Methods:** Swap between standard **Similarity Search (Top-$K$)** and **Maximum Marginal Relevance (MMR)**.
- [X] **$K$-Value:** Change your `k` parameter (the number of chunks retrieved) from $K=1$ to $K=5$. See if adding more pages helps or just adds confusing noise.

#### **6. Generate Response**
- [X] **Guardrails:** Test a loose prompt versus a strict **Grounded Prompt** (e.g., *"If the context does not explicitly mention the answer, reply with 'Data Not Found'"*).
- [X] **Model Temperature:** Change your model's temperature from `0.0` (strict fact extraction) to `1.0` (creative phrasing) and watch for hallucinations.

---

### **7. Run Multiple Experiments**

Run the cells in your notebook using these two tricky student questions designed to stress-test your system:
* **The Normal Question:** *"What happens if I miss an exam because of a family emergency?"* (This info should exist in the handbook).
* **The Out-of-Bounds Question:** *"Where can I buy a parking pass for the campus garage?"* (Assuming parking rules are completely missing from the handbook).

#### **Example Experiment Logs to Replicate:**
* **Experiment A (Loose Setup):** Basic Similarity ($K=1$), Temperature = $0.9$.
  * *Result:* The bot completely hallucinates a fake rule about campus parking passes because it guessed.
* **Experiment B (Production Setup):** MMR Search ($K=3$), Temperature = $0.0$, Strict Grounding Prompt.
  * *Result:* The bot correctly stops itself and states: *"I cannot find information regarding parking passes in the provided handbook segments."*

---

### **8. Note Down Your Observations, Analysis, and Conclusions**

Add a markdown cell and answer the following:

1. **What worked?** Did changing the retrieval method to MMR or adjusting the $K$-value make your answers more complete?
2. **What went wrong?** Show an example of an experiment where the bot hallucinated or cut off a policy answer midway through execution. Why did it happen?
3. **Your Final Settings:** Based on your testing, what are the perfect settings (Chunk Size, Retrieval Type, $K$-value, and Temperature) to deploy a safe, zero-hallucination FAQ Bot for your university?

In [67]:
# Re-examining a document loaded by PyPDFLoader
print("--- Sample Document from PyPDFLoader ---")
print(docs_pdf[0].page_content[:500]) # Print first 500 characters of the first PDF document
print("\nMetadata:", docs_pdf[0].metadata)

--- Sample Document from PyPDFLoader ---
OAKRIDGE ACADEMY
Student & Parent Handbook
Academic Year 2026-2027
Nurturing Excellence, Integrity, and Community
100 Academy Way, Oakridge Valley
www.oakridgeacademy.edu
OA
Oakridge Academy Student Handbook 2026-2027 Page 1

Metadata: {'producer': 'WeasyPrint 62.3', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Oakridge Academy Student Handbook', 'source': 'school_handbook.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [68]:
from langchain_community.document_loaders import WebBaseLoader

web_url = "https://Google.com"
print(f"Loading content from: {web_url}")
loader_web_new = WebBaseLoader(web_url)
docs_web_new = loader_web_new.load()

print("--- Sample Document from WebBaseLoader ---")
if docs_web_new:
    print(docs_web_new[0].page_content[:500]) # Print first 500 characters of the first web document
    print("\nMetadata:", docs_web_new[0].metadata)
else:
    print("Failed to load web content.")

Loading content from: https://Google.com
--- Sample Document from WebBaseLoader ---
GoogleGmail圖片登入 進階搜尋Google 的其他語言版本：  English 廣告商業解決方案關於 GoogleGoogle.com.tw© 2026 - 隱私權 - 服務條款Google 應用程式

Metadata: {'source': 'https://Google.com', 'title': 'Google', 'language': 'zh-TW'}


EXPERIMENTS

In [69]:
# --- TEST QUESTIONS ---
q_normal = "What happens if I miss an exam because of a family emergency?"
q_out_of_bounds = "Where can I buy a parking pass for the campus garage?"

In [73]:
print("--- EXPERIMENT A.1: LOOSE SETUP: SIMILARITY SEARCH (K=1), TEMPERATURE = 0.9 ---")
bad_retriever_k1_temp09 = db.as_retriever(search_type="similarity", search_kwargs={"k": 1})
llm_loose_temp09 = Ollama(model="gemma3:1b", temperature=0.9)

loose_prompt = ChatPromptTemplate.from_template("""
Answer the question using this context:
{context}

Question: {question}
""")
loose_chain_k1_temp09 = loose_prompt | llm_loose_temp09 | StrOutputParser()

# Test Out-of-Bounds Question
bad_context_k1_temp09_oob = "\n".join([doc.page_content for doc in bad_retriever_k1_temp09.invoke(q_out_of_bounds)])
print(f"Q: {q_out_of_bounds}")
print(f"Loose Bot Answer (K=1, Temp=0.9):\n{loose_chain_k1_temp09.invoke({'context': bad_context_k1_temp09_oob, 'question': q_out_of_bounds})}\n")

# Test Normal Question
bad_context_k1_temp09_normal = "\n".join([doc.page_content for doc in bad_retriever_k1_temp09.invoke(q_normal)])
print(f"Q: {q_normal}")
print(f"Loose Bot Answer (K=1, Temp=0.9):\n{loose_chain_k1_temp09.invoke({'context': bad_context_k1_temp09_normal, 'question': q_normal})}\n")

--- EXPERIMENT A.1: LOOSE SETUP: SIMILARITY SEARCH (K=1), TEMPERATURE = 0.9 ---
Q: Where can I buy a parking pass for the campus garage?
Loose Bot Answer (K=1, Temp=0.9):
Please consult the Oak Ridge Academy Student Handbook, specifically page 4, to find out where you can purchase a parking pass for the campus garage. (Details will be in that document.)

Q: What happens if I miss an exam because of a family emergency?
Loose Bot Answer (K=1, Temp=0.9):
Repeated occurrences trigger formal suspension hearings before the Honor Counsel.



In [74]:

print("\n--- EXPERIMENT A.2: LOOSE SETUP: SIMILARITY SEARCH (K=5), TEMPERATURE = 0.9 ---")
bad_retriever_k5_temp09 = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})
loose_chain_k5_temp09 = loose_prompt | llm_loose_temp09 | StrOutputParser()

# Test Out-of-Bounds Question
bad_context_k5_temp09_oob = "\n".join([doc.page_content for doc in bad_retriever_k5_temp09.invoke(q_out_of_bounds)])
print(f"Q: {q_out_of_bounds}")
print(f"Loose Bot Answer (K=5, Temp=0.9):\n{loose_chain_k5_temp09.invoke({'context': bad_context_k5_temp09_oob, 'question': q_out_of_bounds})}\n")

# Test Normal Question
bad_context_k5_temp09_normal = "\n".join([doc.page_content for doc in bad_retriever_k5_temp09.invoke(q_normal)])
print(f"Q: {q_normal}")
print(f"Loose Bot Answer (K=5, Temp=0.9):\n{loose_chain_k5_temp09.invoke({'context': bad_context_k5_temp09_normal, 'question': q_normal})}\n")



--- EXPERIMENT A.2: LOOSE SETUP: SIMILARITY SEARCH (K=5), TEMPERATURE = 0.9 ---
Q: Where can I buy a parking pass for the campus garage?
Loose Bot Answer (K=5, Temp=0.9):
The text doesn’t specifically explain where to purchase a parking pass. It focuses on the uniform, daily attire requirements, and health procedures.

Q: What happens if I miss an exam because of a family emergency?
Loose Bot Answer (K=5, Temp=0.9):
According to the document provided, students must strictly adhere to the infrastructure outlined below if they miss an exam due to a familyemergency:

*   Medication Class Storage Obligation Documentation Required
*   Over-the-Counter (OTC) Locked in Health Suite OnlyParent / Guardian Consent
*   Emergency Inhaler / Epi-PensSelf-Carry Authorized Dual Doctor & Parent Waiver

The document does not give further details regarding the consequences of this situation.



In [75]:
print("\n--- EXPERIMENT A.3: LOOSE SETUP: MMR SEARCH (K=1), TEMPERATURE = 0.0 ---")
bad_retriever_k1_temp00 = db.as_retriever(search_type="mmr", search_kwargs={"k": 1, "fetch_k": 10})
llm_loose_temp00 = Ollama(model="gemma3:1b", temperature=0.0)
loose_chain_k1_temp00 = loose_prompt | llm_loose_temp00 | StrOutputParser()

# Test Out-of-Bounds Question
bad_context_k1_temp00_oob = "\n".join([doc.page_content for doc in bad_retriever_k1_temp00.invoke(q_out_of_bounds)])
print(f"Q: {q_out_of_bounds}")
print(f"Loose Bot Answer (K=1, Temp=0.0):\n{loose_chain_k1_temp00.invoke({'context': bad_context_k1_temp00_oob, 'question': q_out_of_bounds})}\n")

# Test Normal Question
bad_context_k1_temp00_normal = "\n".join([doc.page_content for doc in bad_retriever_k1_temp00.invoke(q_normal)])
print(f"Q: {q_normal}")
print(f"Loose Bot Answer (K=1, Temp=0.0):\n{loose_chain_k1_temp00.invoke({'context': bad_context_k1_temp00_normal, 'question': q_normal})}\n")


--- EXPERIMENT A.3: LOOSE SETUP: MMR SEARCH (K=1), TEMPERATURE = 0.0 ---
Q: Where can I buy a parking pass for the campus garage?
Loose Bot Answer (K=1, Temp=0.0):
According to the handbook, you can purchase a parking pass at the Campus Garage located at 100 Academy Way.

Q: What happens if I miss an exam because of a family emergency?
Loose Bot Answer (K=1, Temp=0.0):
According to the Oakridge Academy Student Handbook, if you miss an examination due to a family emergency, formal suspension hearings will be held before the Honor Council.



In [76]:
print("EXPERIMENT A.4: LOOSE SETUP: MMR (K=5), TEMPERATURE = 0.0")
bad_retriever_mmr_k5_temp00 = db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 10})
llm_loose_mmr_k5_temp00 = Ollama(model="gemma3:1b", temperature=0.0)
loose_chain_mmr_k5_temp00 = loose_prompt | llm_loose_mmr_k5_temp00 | StrOutputParser()

# Test Out-of-Bounds Question
bad_context = "\n".join([doc.page_content for doc in bad_retriever_mmr_k5_temp00.invoke(q_out_of_bounds)])
print(f"Q: {q_out_of_bounds}")
print(f"Loose Bot Answer:\n{loose_chain_mmr_k5_temp00.invoke({'context': bad_context, 'question': q_out_of_bounds})}\n")


# Test Normal Question
bad_context_normal = "\n".join([doc.page_content for doc in bad_retriever_mmr_k5_temp00.invoke(q_normal)])
print(f"Q: {q_normal}")
print(f"Loose Bot Answer:\n{loose_chain_mmr_k5_temp00.invoke({'context': bad_context_normal, 'question': q_normal})}\n")

EXPERIMENT A.4: LOOSE SETUP: MMR (K=5), TEMPERATURE = 0.0
Q: Where can I buy a parking pass for the campus garage?
Loose Bot Answer:
The provided text does not contain information about where to purchase a parking pass for the campus garage. It focuses solely on outlining Oakridge Academy’s student handbook and academic policies.

Q: What happens if I miss an exam because of a family emergency?
Loose Bot Answer:
According to the provided text, students requiring medical updates or medication administration during active operating hours must strictly adhere to the infrastructure outlined below:

Medication Class Storage Obligation Documentation RequiredPrescription Drugs Locked in Health Suite OnlyPhysician Signature Form
Over-the-Counter (OTC) Locked in Health Suite OnlyParent / Guardian ConsentEmergency Inhalers / Epi-PensSelf-Carry Authorized Dual Doctor & Parent Waiver



In [80]:
print("EXPERIMENT B: PRODUCTION SETUP: MMR SEARCH (K=3), TEMPERATURE = 0.0")
good_retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

llm_strict = Ollama(model="gemma3:1b", temperature=0.0)

strict_prompt = ChatPromptTemplate.from_template("""
You are a strict academic assistant. Answer the user's question using the provided context.
If the context does not explicitly mention the answer, reply with 'Data Not Found'

---
RETRIEVED CONTEXT:
{context}

---
USER QUESTION:
{question}

ANSWER:
""")
strict_chain = strict_prompt | llm_strict | StrOutputParser()

# Test Out-of-Bounds Question
good_context_oob = "\n".join([doc.page_content for doc in good_retriever.invoke(q_out_of_bounds)])
print(f"Q: {q_out_of_bounds}")
print(f"Production Bot Answer:\n{strict_chain.invoke({'context': good_context_oob, 'question': q_out_of_bounds})}\n")

# Test Normal Question
good_context_normal = "\n".join([doc.page_content for doc in good_retriever.invoke(q_normal)])
print(f"Q: {q_normal}")
print(f"Production Bot Answer:\n{strict_chain.invoke({'context': good_context_normal, 'question': q_normal})}\n")

EXPERIMENT B: PRODUCTION SETUP: MMR SEARCH (K=3), TEMPERATURE = 0.0
Q: Where can I buy a parking pass for the campus garage?
Production Bot Answer:
Data Not Found

Q: What happens if I miss an exam because of a family emergency?
Production Bot Answer:
Data Not Found




1.   **What worked? Did changing the retrieval method to MMR or adjusting the  K -value make your answers more complete?**

Changing the retrieval method to MMR and adjusting the K-value made the responses more complete. MMR helps balance between similarity and coverage, while a moderate K-value allowed the model to retrieve more chunks.

2.   **What went wrong? Show an example of an experiment where the bot hallucinated or cut off a policy answer midway through execution. Why did it happen?**

In the loose retrieval setup, the bot hallucinated and generated incorrect information. This happened because no context was given and the  prompt was minimal, with no rules regarding unknown information.

3. **Your Final Settings: Based on your testing, what are the perfect settings (Chunk Size, Retrieval Type,  K -value, and Temperature) to deploy a safe, zero-hallucination FAQ Bot for your university?**

*   Chunk Size = automatically set by `SemanticChunker`
*   Retrieval Type = MMR
*   K-value = 3
*   Temperature = 0.0 (to ensure that it is deterministic)




## Section 12. Completion Checklist
To ensure you haven't missed anything, mark the following checkboxes:

- [X] **LLM Limitations**
  - [X] Knowledge boundary demo
  - [X] Hallucination demo

- [X] **Document Ingestion**
  - [X] Unstructured Documents (PDFs)
  - [X] Web Scraping (URLs)

- [X] **Naive / Keyword Search**
  - [X] Keyword extraction & block scanning
  - [X] Synonym failure case

- [X] **Cosine Similarity**
  - [X] Word-pair similarity
  - [X] Sentence-level similarity

- [X] **Chunking Strategies**
  - [X] Fixed-Size
  - [X] Sentence-Based / Header-Based
  - [X] Semantic Chunking

- [X] **Embedding Models**

- [X] **Vector Storage**

- [X] **Advanced Information Retrieval**
  - [X] Similarity Search
  - [X] Top-K Selection
  - [X] Maximum Marginal Relevance (MMR)
  - [X] Context Window Retrieval

- [X] **Generation**

- [X] **University FAQ bot Use-Case**
  - [X] Ingest
  - [X] Chunk
  - [X] Embed
  - [X] Store
  - [X] Retrieve
  - [X] Generate Response
  - [X] Run multiple experiments
  - [X] Note down your observations, analysis and conclusions


## Section 13. Week 5 HandoffThis notebook's final RAG system becomes the retrieval channel for the Week 5 dual-channel bot. The Week 4 notebook layers on memory, guardrails, Streamlit, FastAPI, and request logging.

In [ ]:
# Export the final Week 3 RAG contract for the shared Week 5 delivery layer.week5_rag_contract = {    'retriever': good_retriever,    'prompt': strict_prompt,    'llm': llm_strict,    'chain': strict_chain,}print("Week 3 RAG contract is ready for Week 5 integration")